In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, accuracy_score,
    precision_score, recall_score, f1_score
)
import matplotlib.pyplot as plt

In [2]:
dataset_dir = r"C:/Users/RTX2080Ti/Desktop/AD Connectivity/Images/Eyes Open"
img_height, img_width = 64, 64
batch_size = 32
epochs = 15
seed = 42

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    labels="inferred",
    label_mode="categorical",
    batch_size=batch_size,
    image_size=(img_height, img_width),
    validation_split=0.3,       # 70/30 split
    subset="training",
    seed=seed
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    labels="inferred",
    label_mode="categorical",
    batch_size=batch_size,
    image_size=(img_height, img_width),
    validation_split=0.3,
    subset="validation",
    seed=seed
)

num_classes = len(train_ds.class_names)
class_labels = train_ds.class_names
print("Detected classes:", class_labels)

# Normalize pixel values (0-1)
def normalize_images(image, label):
    return tf.cast(image, tf.float32) / 255.0, label

train_ds = train_ds.map(normalize_images)
test_ds = test_ds.map(normalize_images)

# Convert to arrays (for KFold)
X, y = [], []
for images, labels in train_ds:
    X.extend(images.numpy())
    y.extend(labels.numpy())
X, y = np.array(X), np.array(y)
print("Training data shape:", X.shape, y.shape)

Found 7520 files belonging to 3 classes.
Using 5264 files for training.
Found 7520 files belonging to 3 classes.
Using 2256 files for validation.
Detected classes: ['AD', 'Control', 'FTD']
Training data shape: (5264, 64, 64, 3) (5264, 3)


In [4]:
def build_forcnn_classifier(input_shape=(64, 64, 3), num_classes=3):
    inputs = layers.Input(shape=input_shape)

    def residual_block(x, filters):
        shortcut = x
        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.add([x, shortcut])
        x = layers.ReLU()(x)
        return x

    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = residual_block(x, 32)
    x = layers.Conv2D(64, (3, 3), strides=2, padding='same')(x)
    x = layers.ReLU()(x)
    x = residual_block(x, 64)
    x = layers.Conv2D(128, (3, 3), strides=2, padding='same')(x)
    x = layers.ReLU()(x)
    x = residual_block(x, 128)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name="ForCNN_RGB_CV")
    return model

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=seed)
cv_acc, cv_prec, cv_rec, cv_f1 = [], [], [], []

fold = 1
for train_idx, val_idx in kf.split(X):
    print(f"\n===== Fold {fold} =====")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = build_forcnn_classifier((img_height, img_width, 3), num_classes)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=epochs, batch_size=batch_size, verbose=1)

    y_pred = model.predict(X_val)
    y_true_cls = np.argmax(y_val, axis=1)
    y_pred_cls = np.argmax(y_pred, axis=1)

    acc = accuracy_score(y_true_cls, y_pred_cls)
    prec = precision_score(y_true_cls, y_pred_cls, average='macro', zero_division=0)
    rec = recall_score(y_true_cls, y_pred_cls, average='macro', zero_division=0)
    f1 = f1_score(y_true_cls, y_pred_cls, average='macro', zero_division=0)

    cv_acc.append(acc)
    cv_prec.append(prec)
    cv_rec.append(rec)
    cv_f1.append(f1)
    print(f"Fold {fold} → Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")
    fold += 1

print("\n===== 5-Fold Average Results =====")
print(f"Accuracy: {np.mean(cv_acc):.4f}")
print(f"Precision: {np.mean(cv_prec):.4f}")
print(f"Recall: {np.mean(cv_rec):.4f}")
print(f"F1-score: {np.mean(cv_f1):.4f}")


===== Fold 1 =====
Epoch 1/15
132/132 ━━━━━━━━━━━━━━━━━━━━ 40s 268ms/step - accuracy: 0.4150 - loss: 1.0883 - val_accuracy: 0.3533 - val_loss: 1.1161
Epoch 2/15
132/132 ━━━━━━━━━━━━━━━━━━━━ 34s 260ms/step - accuracy: 0.6310 - loss: 0.8383 - val_accuracy: 0.3523 - val_loss: 1.3722
Epoch 3/15
132/132 ━━━━━━━━━━━━━━━━━━━━ 34s 257ms/step - accuracy: 0.7378 - loss: 0.6336 - val_accuracy: 0.6334 - val_loss: 0.8557
Epoch 4/15
132/132 ━━━━━━━━━━━━━━━━━━━━ 33s 253ms/step - accuracy: 0.8090 - loss: 0.4751 - val_accuracy: 0.4046 - val_loss: 1.4020
Epoch 5/15
132/132 ━━━━━━━━━━━━━━━━━━━━ 33s 247ms/step - accuracy: 0.8468 - loss: 0.3888 - val_accuracy: 0.6752 - val_loss: 0.9532
Epoch 6/15
132/132 ━━━━━━━━━━━━━━━━━━━━ 34s 256ms/step - accuracy: 0.9086 - loss: 0.2492 - val_accuracy: 0.7740 - val_loss: 0.7740
Epoch 7/15
132/132 ━━━━━━━━━━━━━━━━━━━━ 33s 248ms/step - accuracy: 0.9146 - loss: 0.2366 - val_accuracy: 0.7189 - val_loss: 0.8878
Epoch 8/15
132/132 ━━━━━━━━━━━━━━━━━━━━ 33s 252ms/step - accura

In [6]:
print("\n===== Hold-Out Testing on 30% unseen data =====")
final_model = build_forcnn_classifier((img_height, img_width, 3), num_classes)
final_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
final_model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=1)

# Evaluate on hold-out
Y_pred = final_model.predict(test_ds)
y_pred_cls = np.argmax(Y_pred, axis=1)
y_true_cls = np.concatenate([np.argmax(label.numpy(), axis=1) for _, label in test_ds])

print("\nClassification Report (Hold-Out):\n")
print(classification_report(y_true_cls, y_pred_cls, target_names=class_labels))

print("\nConfusion Matrix (Hold-Out):\n")
print(confusion_matrix(y_true_cls, y_pred_cls))


===== Hold-Out Testing on 30% unseen data =====
Epoch 1/15
165/165 ━━━━━━━━━━━━━━━━━━━━ 51s 279ms/step - accuracy: 0.4814 - loss: 1.0418
Epoch 2/15
165/165 ━━━━━━━━━━━━━━━━━━━━ 48s 289ms/step - accuracy: 0.6741 - loss: 0.7535
Epoch 3/15
165/165 ━━━━━━━━━━━━━━━━━━━━ 44s 267ms/step - accuracy: 0.7928 - loss: 0.5353
Epoch 4/15
165/165 ━━━━━━━━━━━━━━━━━━━━ 39s 233ms/step - accuracy: 0.8372 - loss: 0.4154
Epoch 5/15
165/165 ━━━━━━━━━━━━━━━━━━━━ 39s 234ms/step - accuracy: 0.8846 - loss: 0.3073
Epoch 6/15
165/165 ━━━━━━━━━━━━━━━━━━━━ 39s 233ms/step - accuracy: 0.9004 - loss: 0.2600
Epoch 7/15
165/165 ━━━━━━━━━━━━━━━━━━━━ 39s 234ms/step - accuracy: 0.9192 - loss: 0.2139
Epoch 8/15
165/165 ━━━━━━━━━━━━━━━━━━━━ 39s 238ms/step - accuracy: 0.9401 - loss: 0.1659
Epoch 9/15
165/165 ━━━━━━━━━━━━━━━━━━━━ 39s 235ms/step - accuracy: 0.9433 - loss: 0.1571
Epoch 10/15
165/165 ━━━━━━━━━━━━━━━━━━━━ 39s 234ms/step - accuracy: 0.9564 - loss: 0.1143
Epoch 11/15
165/165 ━━━━━━━━━━━━━━━━━━━━ 39s 238ms/step - ac